# DIABETES PREDICTION USING LOGISTIC REGRESSION
### Machine Learning Mini Project

**Submitted by:** Poorvi M S  
**Dataset:** Pima Indians Diabetes Database (Kaggle)  
**Algorithm:** Balanced Logistic Regression  

---

## Project overview
This notebook develops a binary classification model to estimate whether a record belongs to the diabetes-positive or diabetes-negative class using eight diagnostic input features. The target column is `Outcome` (`0` = No diabetes, `1` = Diabetes).

**Important:** This is an educational machine-learning exercise. It is not a medical diagnostic tool and must not be used to make healthcare decisions.

### Dataset download
Kaggle dataset page: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

Download the dataset and upload the file named **`diabetes.csv`** when prompted in the next section. In Google Colab, use the folder/upload icon or the upload prompt generated by the code.

## 1. Import required libraries
We use Pandas and NumPy for data handling, Matplotlib and Seaborn for visualisation, and Scikit-learn for preprocessing, model training and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## 2. Load the Kaggle dataset
Upload `diabetes.csv` from the downloaded Kaggle dataset. The code checks that the expected columns are present before continuing.

In [ ]:
uploaded = files.upload()
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Please upload the Kaggle CSV file named diabetes.csv.")

df = pd.read_csv(csv_files[0])
expected_columns = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin",
    "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]
missing_columns = [col for col in expected_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing expected columns: {missing_columns}")

df = df[expected_columns].copy()
print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


## 3. Understand and inspect the data
The first inspection checks the number of rows and columns, data types, missing values and descriptive statistics. This helps identify data-quality issues before training.

In [ ]:
print("Dataset information:")
df.info()

print("\\nMissing values in each column:")
display(df.isnull().sum().to_frame("Missing values"))

print("\\nDescriptive statistics:")
display(df.describe().T)


## 4. Exploratory Data Analysis (EDA)
The target distribution shows how many records belong to each class. Histograms help inspect feature distributions, while the correlation heatmap shows pairwise linear relationships. These visualisations support understanding of the dataset; correlation does not imply causation.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Outcome")
plt.title("Distribution of Diabetes Outcome")
plt.xlabel("Outcome (0 = No diabetes, 1 = Diabetes)")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

df.hist(figsize=(14, 10), bins=20, edgecolor="black")
plt.suptitle("Feature Distributions", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


## 5. Handle invalid zero values
In this dataset, zero values in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI` are treated as invalid/missing measurements for this exercise. Zero is retained for `Pregnancies` because it is a valid value. We convert the selected zeros to `NaN`; median imputation is then learned from the training data only to avoid data leakage.

In [ ]:
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df_model = df.copy()
df_model[zero_as_missing] = df_model[zero_as_missing].replace(0, np.nan)

print("Missing/invalid values after zero replacement:")
display(df_model.isnull().sum().to_frame("Missing or invalid values"))

X = df_model.drop(columns="Outcome")
y = df_model["Outcome"].astype(int)

print("Target class counts:")
display(y.value_counts().sort_index().rename(index={0: "No diabetes (0)", 1: "Diabetes (1)"}))


## 6. Split the dataset into training and testing sets
The data is divided into 80% training and 20% testing records. Stratification preserves approximately the same class proportions in both sets. A fixed random state makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])


## 7. Build the preprocessing and Logistic Regression pipeline
The pipeline performs two preprocessing operations followed by classification:
- **Median imputation:** fills missing measurements using the median of each training feature.
- **StandardScaler:** standardises feature values so variables measured on different scales are more comparable.
- **Balanced Logistic Regression:** `class_weight='balanced'` adjusts class weights inversely to class frequencies. Logistic Regression estimates the probability of the positive class and predicts a binary outcome.

Using a pipeline ensures that preprocessing is fitted only on the training data and then consistently applied to test data.

In [ ]:
model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000,
                                      random_state=RANDOM_STATE))
])

model.fit(X_train, y_train)
print("Model training completed.")


## 8. Make predictions and evaluate the model
Accuracy measures overall correctness. Precision measures how many predicted positive cases are actually positive. Recall measures how many actual positive cases are detected. F1-score balances precision and recall. ROC-AUC measures ranking/separation ability across classification thresholds.

For a screening-style learning task, recall is important because a false negative is a positive case missed by the model. However, no single metric is sufficient to assess a real clinical system.

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}
metrics_df = pd.DataFrame.from_dict(metrics, orient="index", columns=["Score"])
display(metrics_df.round(3))

print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["No diabetes", "Diabetes"], zero_division=0))


## 9. Confusion matrix
The confusion matrix compares actual labels with predicted labels:
- **TN (True Negative):** actual 0, predicted 0.
- **FP (False Positive):** actual 0, predicted 1.
- **FN (False Negative):** actual 1, predicted 0.
- **TP (True Positive):** actual 1, predicted 1.

The counts below are generated from this notebook's own test split and may differ from the reference values printed in the supplied project brief if the split or software settings differ.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No diabetes (0)", "Diabetes (1)"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")


## 10. ROC curve
The ROC curve plots the true-positive rate against the false-positive rate at different thresholds. The area under this curve (ROC-AUC) summarises the model's ability to rank positive cases above negative cases.

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve - Logistic Regression")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Interpret model coefficients
A positive coefficient indicates that, after scaling and accounting for the other features in the model, an increase in that feature is associated with higher log-odds of the positive class. A negative coefficient indicates the opposite association. Coefficients describe this fitted model and should not be interpreted as proof of causation or as clinical risk estimates.

In [ ]:
classifier = model.named_steps["classifier"]
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": classifier.coef_[0]
}).sort_values("Coefficient", ascending=False)

display(coef_df)

plt.figure(figsize=(9, 5))
sns.barplot(data=coef_df, x="Coefficient", y="Feature")
plt.axvline(0, color="black", linewidth=1)
plt.title("Logistic Regression Coefficients")
plt.xlabel("Coefficient (standardised features)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 12. Final conclusion
This project demonstrates a complete binary-classification workflow using the Pima Indians Diabetes dataset and balanced Logistic Regression. The dataset is inspected, invalid zero measurements are treated as missing, and median imputation and standardisation are applied within a training pipeline. Model performance is evaluated using accuracy, precision, recall, F1-score, ROC-AUC and a confusion matrix.

The actual metric values and plots produced above should be used when writing the final submission conclusion. In the supplied project brief, the reference results are accuracy 0.734, precision 0.603, recall 0.704, F1-score 0.650 and ROC-AUC 0.813, with TN=75, FP=25, FN=16 and TP=38. Your executed notebook may produce different values depending on the exact split and settings. State the values shown by your run.

**Limitation and ethical note:** The dataset is small, historical and represents a specific population. It does not represent everyone. The model is for learning classification concepts only and must not be used for diagnosis, treatment or personal medical decisions.

### Student
**Poorvi M S**

## References
1. UCI Machine Learning / Kaggle. *Pima Indians Diabetes Database*. https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
2. Scikit-learn documentation. *LogisticRegression*. https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
3. Scikit-learn documentation. *Pipeline and preprocessing*. https://scikit-learn.org/stable/modules/compose.html
4. Scikit-learn documentation. *Model evaluation*. https://scikit-learn.org/stable/modules/model_evaluation.html

---
**Submission reminder:** Run all cells in Google Colab, confirm that every output and graph is visible, then download the executed notebook using **File → Download → Download .ipynb**.